# Agon Evaluation Analysis

This notebook compares the full debate system against two baselines on the 25 **philosophical** problems in `data/problems.json`. **All results come from a single real pipeline run** (`results/debate_runs.json`) — nothing is simulated or hardcoded.

- `single_agent_baseline`: one solver's independent, pre-debate answer.
- `majority_vote_baseline`: majority vote over the three solvers' pre-debate answers.
- `full_debate`: the judge's final answer after peer review and refinement.

Answers are graded by an LLM-as-judge grader (`evaluation/metrics.py`) on whether they reach a defensible position and engage each problem's required considerations.

> ⚠️ **Run these first to populate `results/`, then execute this notebook top-to-bottom:**
>
> ```bash
> python main.py                    # -> results/debate_runs.json
> python -m evaluation.baseline     # -> results/baseline_comparison.json
> python -m evaluation.metrics      # -> results/evaluation_*.json / .csv
> python -m evaluation.plots        # -> results/plots/*.png
> ```
>
> (An earlier STEM run is archived in `results/stem_baseline/` — the model scored 100% across all systems there, which motivated the pivot to philosophical problems.)

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import SVG, Image, display

ROOT = Path('..').resolve()
results_dir = ROOT / 'results'
plots_dir = results_dir / 'plots'

summary = json.loads((results_dir / 'evaluation_summary.json').read_text(encoding='utf-8'))
rows = pd.read_csv(results_dir / 'evaluation_rows.csv')
rows.head()

## Dataset Overview

The current dataset contains 25 problems split across the five required categories.

In [ ]:
problem_data = json.loads((ROOT / 'data' / 'problems.json').read_text(encoding='utf-8'))
problems = problem_data['problems'] if isinstance(problem_data, dict) else problem_data
category_table = (
    pd.DataFrame(problems)
    .groupby('category')
    .agg(problem_count=('id', 'count'), first_problem=('id', 'min'), last_problem=('id', 'max'))
    .reset_index()
)
category_table

## Overall Results

The full debate system is compared directly to a single-agent baseline and a majority-vote baseline using accuracy and mean rubric score across all 25 problems.

In [ ]:
system_table = pd.DataFrame(summary['systems']).T
system_table[['n', 'correct', 'accuracy', 'mean_score']]

In [ ]:
print(f"Improvement rate (pre-debate wrong -> full debate correct): {summary['improvement_rate']:.0%}")
print(f"Consensus rate (all 3 solvers' initial answers agree):     {summary['consensus_rate']:.0%}")
n_disagree = summary.get('judge_decision_count', 0)
print(f"Judge accuracy (on the {n_disagree} problems where refined "
      f"solver answers disagree):                                  {summary['judge_accuracy']:.0%}")

## Category Breakdown

In [ ]:
category_rows = []
for category, systems in summary['categories'].items():
    for system, values in systems.items():
        category_rows.append({'category': category, 'system': system, **values})
pd.DataFrame(category_rows).sort_values(['category', 'system'])

## Generated Plots

The plots below are generated by `python -m evaluation.plots`. In environments with matplotlib they are saved as PNG files; otherwise the script writes SVG fallbacks.

In [ ]:
for stem in ['system_accuracy', 'category_scores', 'difficulty_accuracy', 'debate_rates']:
    png = plots_dir / f'{stem}.png'
    svg = plots_dir / f'{stem}.svg'
    print(stem)
    if png.exists():
        display(Image(filename=str(png)))
    elif svg.exists():
        display(SVG(filename=str(svg)))

## Interpretation

The evaluation uses 25 mixed problems across five categories: mathematical/logical reasoning, physics/scientific reasoning, logic puzzles or constraint satisfaction, strategic/game-theory reasoning, and gradeable philosophical/ethical reasoning.

All numbers above are computed from a single real run of the debate pipeline (`results/debate_runs.json`) — the three systems are derived from that run and every answer is graded by an LLM-as-judge grader against the reference answer. Read the tables and plots above for the actual outcome on this run:

- **Overall accuracy** shows whether full debate beats the single-agent and majority-vote baselines.
- **Improvement rate** counts problems where the pre-debate majority answer was wrong but the debate's final answer was correct — i.e. where peer review + refinement fixed a mistake.
- **Consensus rate** is how often all three solvers independently reached the same answer before any debate.
- **Judge accuracy** is measured only on the problems where the refined solver answers disagreed: on those, did the judge pick a correct final answer?

Because the results are measured rather than assumed, debate may not win on every metric or category — the point of the experiment is the honest comparison. To re-run end-to-end, execute the four commands listed at the top of this notebook.